# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashalaf/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression first, then Random Forest**, per the skill's own table: this is a yes/no question with an observed label (`is_declining_label`), and the deliverable is a ranked queue evaluated at precision@50, so a classifier's probability output is what actually gets used, not the raw yes/no prediction. Starting simple on purpose, a readable model that can be beaten by something stronger is a real result; a complex model that can't be explained isn't automatically a better one.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yashalaf/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), \
    "starter CSV not found -- are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

static_numeric = ["word_count", "char_count", "search_volume", "competition", "cpc",
                   "content_age_days", "days_since_last_update"]
static_categorical = ["content_type", "main_intent", "competition_level", "age_tier", "freshness_tier"]
safe_traffic = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]

numeric_cols = static_numeric + safe_traffic
X_numeric = df[numeric_cols].fillna(-1)
X_categorical = pd.get_dummies(df[static_categorical].fillna("unknown"), prefix=static_categorical)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]

print("feature matrix:", X.shape)

feature matrix: (30000, 30)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`, not random, and not by why-it-sounds-safer, by evidence.** ML-05 already measured this directly: a random split scored 0.644 AUC while a client-grouped split scored 0.551, a 0.093 gap that was the model partly memorizing clients it had already seen, not genuine signal. A random split here would repeat that mistake. `GroupShuffleSplit` guarantees zero client overlap between train and test, confirmed below rather than assumed.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
test_df = df.iloc[test_idx].copy()

client_overlap = set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])
print(f"train rows: {len(train_idx)}, test rows: {len(test_idx)}")
print(f"client overlap between train and test (must be 0): {len(client_overlap)}")

train rows: 23837, test rows: 6163
client overlap between train and test (must be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Same test rows, same metric (precision@50), for the baseline and every model.** The ML-07 baseline rule is recomputed here on this exact held-out test set, not copy-pasted from Week 4's whole-dataset number, that earlier number was never evaluated on a genuinely unseen client split, so it isn't a fair comparison point until it's rerun this same way.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50

# baseline rule, recomputed on the test split only (same rule as ML-07)
visible = (test_df["impressions_90d"] >= 500).astype(int)
stale = (test_df["days_since_last_update"] >= 90).astype(int)
weak_position = ((test_df["avg_position"] > 0) & (test_df["avg_position"] >= 15)).astype(int)
test_df["baseline_score"] = visible * stale * weak_position * test_df["impressions_90d"]

baseline_p50 = precision_at_k(test_df["baseline_score"].values, yte.values, K)
dummy_p50 = precision_at_k(test_df["impressions_90d"].values, yte.values, K)
base_rate = yte.mean()

# Logistic Regression
scaler = StandardScaler()
Xtr_scaled = scaler.fit_transform(Xtr)
Xte_scaled = scaler.transform(Xte)
lr = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
lr.fit(Xtr_scaled, ytr)
lr_p50 = precision_at_k(lr.predict_proba(Xte_scaled)[:, 1], yte.values, K)

# Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(Xtr, ytr)
rf_p50 = precision_at_k(rf.predict_proba(Xte)[:, 1], yte.values, K)

comparison = pd.DataFrame({
    "method": ["base rate (guessing)", "dummy (sort by impressions)", "ML-07 rule baseline",
               "Logistic Regression", "Random Forest"],
    f"precision@{K}": [base_rate, dummy_p50, baseline_p50, lr_p50, rf_p50],
})
print(comparison.to_string(index=False))

                     method  precision@50
       base rate (guessing)      0.510952
dummy (sort by impressions)      0.440000
        ML-07 rule baseline      0.320000
        Logistic Regression      0.600000
              Random Forest      0.480000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**The simpler model won, and that's the actual finding, not a disappointing result to explain away.** Logistic Regression beat both baselines and the base rate; Random Forest, despite being the "stronger" default per the skill's own table, did not. Reported as found, per the skill's own instruction not to bury a result that contradicts the expected escalation.

In [4]:
# Feature importances from Random Forest, sanity-checked, not just reported
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("top 5 feature importances:")
print(importances.head(5))
print(f"\ntop feature is {importances.index[0]} at {importances.iloc[0]:.1%} of total importance, "
      f"not suspiciously dominant (a single feature near 90-100% would be the leakage red flag from ML-05), "
      f"and it's a plausible driver: recent traffic volume relating to whether a page later declines makes sense.")

top 5 feature importances:
impressions_prev_30d    0.431900
content_age_days        0.113345
word_count              0.067618
sessions_prev_30d       0.061299
char_count              0.054831
dtype: float64

top feature is impressions_prev_30d at 43.2% of total importance, not suspiciously dominant (a single feature near 90-100% would be the leakage red flag from ML-05), and it's a plausible driver: recent traffic volume relating to whether a page later declines makes sense.


In [5]:
# 3 concrete wrong cases from the better model (Logistic Regression)
test_df["pred_proba"] = lr.predict_proba(Xte_scaled)[:, 1]
test_df["actual"] = yte.values

print("False positives: flagged as high-risk, but not actually declining")
top50 = test_df.sort_values("pred_proba", ascending=False).head(50)
false_pos = top50[top50["actual"] == 0].head(2)
print(false_pos[["content_id", "pred_proba", "impressions_prev_30d", "content_age_days", "main_intent"]].to_string(index=False))

print("\nFalse negative: actually declining, but scored very low")
false_neg = test_df[test_df["actual"] == 1].sort_values("pred_proba").head(1)
print(false_neg[["content_id", "pred_proba", "impressions_prev_30d", "content_age_days", "main_intent"]].to_string(index=False))

False positives: flagged as high-risk, but not actually declining
          content_id  pred_proba  impressions_prev_30d  content_age_days   main_intent
content_d0cadc3e2773    0.843278                 21552               105 informational
content_c84a0ab98e90    0.837452                 84773                95 informational

False negative: actually declining, but scored very low
          content_id  pred_proba  impressions_prev_30d  content_age_days   main_intent
content_9532f197bbc8    0.010046                174235               445 informational


**Why these are hard, in plain words:**

- The two false positives both have solid recent traffic (21K-85K impressions) and are only ~100 days old, exactly the profile the model learned to associate with risk, but they happen to be holding steady. The model can't see anything about content quality or competitor movement, only traffic and age, so a page that's simply stable-but-traffic-heavy looks identical to one that's about to decline.
- The false negative is the harder pattern: a 445-day-old page with 174K prior-30d impressions, scored a near-zero decline probability, and it's actually declining. High sustained traffic on an old page reads as "established and fine" to the model, but established pages decline too, and the model has no signal here (no ranking or CTR data survived the leakage cut in ML-05) to catch a slow position slide before it shows up in traffic.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.